# Sistema multiagente con LangChain para gestión de tickets de soporte — EduFlowTech

**EduFlowTech** es una empresa ficticia especializada en plataformas educativas en línea. En este notebook se implementa un flujo de trabajo con **tres agentes** construidos con LangChain que colaboran para procesar consultas de soporte de los usuarios sobre cursos, lecciones y ejercicios.

## Arquitectura del sistema

| Agente | Función |
|---|---|
| **Agente 1 — Procesador de consultas** | Interpreta la pregunta del usuario, la clasifica (lección / ejercicio / bug) y extrae el tema principal. |
| **Agente 2 — Buscador de contenido** | Consulta la base de datos simulada de cursos/lecciones/ejercicios y encuentra el contenido más relevante para el tema detectado. |
| **Agente 3 — Generador de respuestas** | Toma la clasificación del Agente 1 y el contenido encontrado por el Agente 2, y redacta una respuesta personalizada para el usuario. |

El flujo de datos es: **Usuario → Agente 1 → Agente 2 → Agente 3 → Usuario**.

## 1. Preparar el entorno

In [ ]:
!pip install --upgrade langchain langchain-openai langchain-core openai pandas matplotlib pydantic

In [ ]:
import json
from datetime import datetime, timezone
from enum import Enum
from typing import Optional

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

import openai

## 2. Diagrama del flujo de agentes

Visualizamos cómo fluyen los datos entre el usuario, los tres agentes y la base de datos.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
ax.axis("off")

boxes = {
    "Usuario": (0.3, 1.6, 1.6, 0.8, "#4C72B0"),
    "Agente 1\nProcesador de\nconsultas": (2.3, 1.6, 1.8, 0.8, "#DD8452"),
    "Agente 2\nBuscador de\ncontenido": (4.6, 1.6, 1.8, 0.8, "#55A868"),
    "Agente 3\nGenerador de\nrespuestas": (6.9, 1.6, 1.8, 0.8, "#C44E52"),
    "Usuario\n(respuesta)": (9.0, 1.6, 1.0, 0.8, "#4C72B0"),
    "Base de datos\n(cursos, lecciones,\nejercicios)": (4.6, 0.1, 1.8, 0.8, "#8172B2"),
}

for label, (x, y, w, h, color) in boxes.items():
    rect = mpatches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.05",
        linewidth=1.5,
        edgecolor="black",
        facecolor=color,
        alpha=0.85,
    )
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center",
            fontsize=9, color="white", weight="bold")

arrows = [
    (1.9, 2.0, 2.3, 2.0),
    (4.1, 2.0, 4.6, 2.0),
    (6.4, 2.0, 6.9, 2.0),
    (8.7, 2.0, 9.0, 2.0),
    (5.5, 1.6, 5.5, 0.9),
]

for x0, y0, x1, y1 in arrows:
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="->", lw=1.5, color="black"))

plt.title("Flujo de agentes — Sistema de soporte EduFlowTech", fontsize=12)
plt.tight_layout()
plt.savefig("agents_workflow_diagram.png", dpi=150)
plt.show()

## 3. Configuración de OpenAI (API Key ficticia)

Se utiliza una API Key ficticia, tal y como se pide en el ejercicio. El sistema incluye un modo `MOCK_MODE` que simula las respuestas de los agentes basados en LLM cuando no se dispone de una clave real, permitiendo ejecutar el notebook de principio a fin. Con una API Key real, basta con poner `MOCK_MODE = False`.

In [ ]:
openai.api_key = "sk-YOUR_FAKE_API_KEY"

MODEL_NAME = "gpt-4o-mini"
MOCK_MODE = True

llm = ChatOpenAI(model=MODEL_NAME, api_key=openai.api_key, temperature=0)

## 4. Base de datos simulada de cursos, lecciones y ejercicios

Creamos un `DataFrame` de `pandas` con columnas `curso`, `tema`, `leccion`, `ejercicio`, `nivel_dificultad` y `link`, que actúa como la base de datos que consultará el Agente 2.

In [ ]:
courses_data = [
    {"curso": "Deep Learning Basico", "tema": "redes neuronales", "leccion": "Introduccion a Redes Neuronales",
     "ejercicio": "Implementar un perceptron simple en Python", "nivel_dificultad": "principiante",
     "link": "https://eduflowtech.com/cursos/deep-learning-basico/leccion-1"},
    {"curso": "Deep Learning Basico", "tema": "redes neuronales convolucionales", "leccion": "Introduccion a CNNs",
     "ejercicio": "Clasificar imagenes con una CNN sencilla", "nivel_dificultad": "intermedio",
     "link": "https://eduflowtech.com/cursos/deep-learning-basico/leccion-4"},
    {"curso": "Fundamentos de Python", "tema": "estructuras de datos", "leccion": "Listas y diccionarios",
     "ejercicio": "Crear una agenda de contactos con diccionarios", "nivel_dificultad": "principiante",
     "link": "https://eduflowtech.com/cursos/fundamentos-python/leccion-3"},
    {"curso": "Fundamentos de Python", "tema": "funciones", "leccion": "Definicion de funciones",
     "ejercicio": "Escribir una funcion que calcule el factorial", "nivel_dificultad": "principiante",
     "link": "https://eduflowtech.com/cursos/fundamentos-python/leccion-5"},
    {"curso": "Machine Learning Aplicado", "tema": "regresion", "leccion": "Regresion lineal y logistica",
     "ejercicio": "Entrenar un modelo de regresion con scikit-learn", "nivel_dificultad": "intermedio",
     "link": "https://eduflowtech.com/cursos/ml-aplicado/leccion-2"},
    {"curso": "Machine Learning Aplicado", "tema": "arboles de decision", "leccion": "Arboles de decision y Random Forest",
     "ejercicio": "Construir un Random Forest para clasificacion", "nivel_dificultad": "intermedio",
     "link": "https://eduflowtech.com/cursos/ml-aplicado/leccion-6"},
    {"curso": "Deep Learning Avanzado", "tema": "transformers", "leccion": "Arquitectura Transformer",
     "ejercicio": "Implementar self-attention desde cero", "nivel_dificultad": "avanzado",
     "link": "https://eduflowtech.com/cursos/deep-learning-avanzado/leccion-1"},
    {"curso": "Deep Learning Avanzado", "tema": "redes neuronales recurrentes", "leccion": "RNNs y LSTM",
     "ejercicio": "Predecir series temporales con una LSTM", "nivel_dificultad": "avanzado",
     "link": "https://eduflowtech.com/cursos/deep-learning-avanzado/leccion-3"},
    {"curso": "Bases de Datos", "tema": "SQL", "leccion": "Consultas SQL basicas",
     "ejercicio": "Escribir consultas SELECT con filtros", "nivel_dificultad": "principiante",
     "link": "https://eduflowtech.com/cursos/bases-de-datos/leccion-2"},
    {"curso": "Bases de Datos", "tema": "modelado de datos", "leccion": "Diseno de esquemas relacionales",
     "ejercicio": "Disenar el esquema de una tienda online", "nivel_dificultad": "intermedio",
     "link": "https://eduflowtech.com/cursos/bases-de-datos/leccion-5"},
    {"curso": "NLP con Transformers", "tema": "procesamiento de lenguaje natural", "leccion": "Tokenizacion y embeddings",
     "ejercicio": "Tokenizar un corpus con la libreria transformers", "nivel_dificultad": "intermedio",
     "link": "https://eduflowtech.com/cursos/nlp-transformers/leccion-2"},
    {"curso": "NLP con Transformers", "tema": "modelos de lenguaje generativos", "leccion": "Fine-tuning de LLMs",
     "ejercicio": "Afinar un modelo pequeno para clasificacion de texto", "nivel_dificultad": "avanzado",
     "link": "https://eduflowtech.com/cursos/nlp-transformers/leccion-6"},
]

courses_db = pd.DataFrame(courses_data)
courses_db

## 5. Definición de los agentes

### Agente 1 — Procesador de consultas

Clasifica la consulta del usuario en una de las categorías `leccion`, `ejercicio` o `bug`, y extrae el tema principal sobre el que pregunta. La salida se estructura con `pydantic` para garantizar un formato consistente que puedan consumir los siguientes agentes.

In [ ]:
class QueryCategory(str, Enum):
    LECCION = "leccion"
    EJERCICIO = "ejercicio"
    BUG = "bug"


class QueryClassification(BaseModel):
    category: QueryCategory = Field(description="Categoria de la consulta del usuario")
    topic: str = Field(description="Tema principal por el que pregunta el usuario, en pocas palabras")
    summary: str = Field(description="Resumen breve de lo que necesita el usuario")


query_processor_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Eres el Agente 1 (Procesador de consultas) del sistema de soporte de EduFlowTech, "
     "una plataforma educativa online. Tu tarea es leer la consulta de un usuario y clasificarla "
     "en una de estas categorias: 'leccion' (pregunta sobre que leccion estudiar), "
     "'ejercicio' (pregunta sobre practicar o resolver ejercicios), o 'bug' (reporte de un error "
     "tecnico en la plataforma). Ademas debes extraer el tema principal de la consulta."),
    ("user", "{query}"),
])


def mock_query_processor(query: str) -> QueryClassification:
    query_lower = query.lower()

    if any(word in query_lower for word in ["error", "bug", "no funciona", "falla", "roto"]):
        category = QueryCategory.BUG
    elif any(word in query_lower for word in ["ejercicio", "practicar", "practica", "resolver"]):
        category = QueryCategory.EJERCICIO
    else:
        category = QueryCategory.LECCION

    known_topics = courses_db["tema"].unique().tolist()
    topic_found = next((t for t in known_topics if t in query_lower), None)

    if topic_found is None:
        if "neuronal" in query_lower:
            topic_found = "redes neuronales"
        elif "python" in query_lower:
            topic_found = "funciones"
        elif "sql" in query_lower or "base de datos" in query_lower:
            topic_found = "SQL"
        else:
            topic_found = query_lower

    return QueryClassification(
        category=category,
        topic=topic_found,
        summary=f"El usuario pregunta sobre '{topic_found}' relacionado con '{category.value}'.",
    )


def run_query_processor_agent(query: str) -> QueryClassification:
    if MOCK_MODE:
        return mock_query_processor(query)

    structured_llm = llm.with_structured_output(QueryClassification)
    chain = query_processor_prompt | structured_llm
    return chain.invoke({"query": query})

### Agente 2 — Buscador de contenido

Recibe el tema extraído por el Agente 1 y busca en la base de datos (`courses_db`) la lección o ejercicio más relevante, usando coincidencia de palabras clave entre el tema y las columnas `tema`, `leccion` y `curso`.

In [ ]:
def run_content_search_agent(classification: QueryClassification) -> Optional[dict]:
    topic_words = set(classification.topic.lower().split())

    def score_row(row) -> int:
        text = f"{row['tema']} {row['leccion']} {row['curso']}".lower()
        return sum(1 for word in topic_words if word in text)

    scores = courses_db.apply(score_row, axis=1)

    if scores.max() == 0:
        return None

    best_row = courses_db.loc[scores.idxmax()]
    return best_row.to_dict()

### Agente 3 — Generador de respuestas

Toma la clasificación del Agente 1 y el contenido encontrado por el Agente 2, y redacta la respuesta final personalizada para el usuario.

In [ ]:
response_generator_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Eres el Agente 3 (Generador de respuestas) del sistema de soporte de EduFlowTech. "
     "Recibes la categoria de la consulta de un usuario y el contenido mas relevante encontrado "
     "en la base de datos de cursos. Redacta una respuesta breve, clara y amable para el usuario, "
     "recomendando la leccion o ejercicio encontrado e incluyendo el enlace proporcionado. "
     "Si no se encontro contenido relevante, indica amablemente que se escalara la consulta al equipo de soporte."),
    ("user",
     "Categoria de la consulta: {category}\n"
     "Tema detectado: {topic}\n"
     "Contenido encontrado: {content}\n\n"
     "Genera la respuesta final para el usuario."),
])


def mock_response_generator(classification: QueryClassification, content: Optional[dict]) -> str:
    if content is None:
        return (
            "No he encontrado contenido especifico sobre ese tema en nuestra plataforma. "
            "He escalado tu consulta al equipo de soporte de EduFlowTech, que se pondra en contacto contigo pronto."
        )

    if classification.category == QueryCategory.BUG:
        return (
            f"Gracias por reportar el problema relacionado con '{classification.topic}'. "
            f"Hemos registrado tu reporte y lo hemos vinculado a la leccion '{content['leccion']}' "
            f"del curso '{content['curso']}' para que el equipo tecnico lo revise. "
            f"Mientras tanto, puedes consultar el material aqui: {content['link']}"
        )

    if classification.category == QueryCategory.EJERCICIO:
        return (
            f"Para practicar sobre '{classification.topic}', te recomendamos el ejercicio "
            f"'{content['ejercicio']}', de nivel {content['nivel_dificultad']}, dentro de la leccion "
            f"'{content['leccion']}' del curso '{content['curso']}'. Aqui tienes el enlace: {content['link']}"
        )

    return (
        f"La leccion recomendada es '{content['leccion']}', del curso '{content['curso']}' "
        f"(nivel {content['nivel_dificultad']}). Aqui tienes el enlace: {content['link']}"
    )


def run_response_generator_agent(classification: QueryClassification, content: Optional[dict]) -> str:
    if MOCK_MODE:
        return mock_response_generator(classification, content)

    chain = response_generator_prompt | llm
    result = chain.invoke({
        "category": classification.category.value,
        "topic": classification.topic,
        "content": json.dumps(content, ensure_ascii=False) if content else "Ninguno encontrado",
    })
    return result.content

## 6. Orquestación del workflow

La función `process_support_ticket` coordina la ejecución secuencial de los tres agentes: Agente 1 → Agente 2 → Agente 3, y devuelve tanto la respuesta final como los resultados intermedios (para trazabilidad).

In [ ]:
def process_support_ticket(user_query: str) -> dict:
    classification = run_query_processor_agent(user_query)
    content = run_content_search_agent(classification)
    final_response = run_response_generator_agent(classification, content)

    return {
        "user_query": user_query,
        "agent_1_classification": classification.model_dump(mode="json"),
        "agent_2_content_found": content,
        "agent_3_final_response": final_response,
        "processed_at": datetime.now(timezone.utc).isoformat(),
    }

## 7. Ejemplo práctico de interacción

Simulamos la consulta del usuario:

> "¿Cuál es la lección más adecuada para aprender sobre redes neuronales?"

In [ ]:
result = process_support_ticket("¿Cual es la leccion mas adecuada para aprender sobre redes neuronales?")

print("--- Agente 1: clasificacion de la consulta ---")
print(json.dumps(result["agent_1_classification"], indent=2, ensure_ascii=False))

print("\n--- Agente 2: contenido encontrado en la base de datos ---")
print(json.dumps(result["agent_2_content_found"], indent=2, ensure_ascii=False))

print("\n--- Agente 3: respuesta final para el usuario ---")
print(result["agent_3_final_response"])

Probamos el sistema con otras consultas para verificar que el flujo funciona correctamente con distintas categorías (ejercicio y bug).

In [ ]:
test_queries = [
    "Necesito un ejercicio para practicar arboles de decision",
    "El video de la leccion de transformers no carga, me da un error",
    "Quiero aprender SQL, que leccion me recomiendas?",
]

for q in test_queries:
    result = process_support_ticket(q)
    print(f"Consulta: {q}")
    print(f"Categoria detectada: {result['agent_1_classification']['category']}")
    print(f"Respuesta: {result['agent_3_final_response']}")
    print("-" * 80)

Probamos también un caso donde no existe contenido relevante en la base de datos, para comprobar que el sistema escala la consulta correctamente en lugar de fallar.

In [ ]:
result_no_match = process_support_ticket("Tengo una duda sobre contabilidad financiera avanzada")

print(f"Categoria detectada: {result_no_match['agent_1_classification']['category']}")
print(f"Contenido encontrado: {result_no_match['agent_2_content_found']}")
print(f"Respuesta: {result_no_match['agent_3_final_response']}")

## Conclusiones

- Se implementó un sistema multiagente con **LangChain** compuesto por tres agentes con funciones claramente diferenciadas: procesamiento/clasificación de la consulta, búsqueda de contenido y generación de la respuesta final.
- Se diseñó y visualizó el flujo de datos entre el usuario, los agentes y la base de datos simulada.
- Se creó una base de datos simulada con `pandas` que representa los cursos, lecciones y ejercicios de EduFlowTech.
- Se implementó un modo `MOCK_MODE` que permite ejecutar y probar todo el flujo multiagente sin necesidad de una API Key real, cumpliendo con el uso de una clave ficticia solicitado en el ejercicio; con `MOCK_MODE = False` y una key real, los Agentes 1 y 3 usarían el LLM de OpenAI a través de LangChain.
- Se probó el sistema con varias consultas de ejemplo, incluyendo el caso de "redes neuronales" solicitado en el enunciado, así como casos de ejercicio, bug y ausencia de contenido relevante, confirmando que el flujo Agente 1 → Agente 2 → Agente 3 funciona correctamente en todos los escenarios.